# Subagent 도구 위임

중앙 에이전트가 전문 에이전트에 필요한 맥락만 전달하면 호출 경계가 어떻게 드러날까요?


## 실행 환경 준비

Python 3.12와 고정된 패키지 환경을 사용합니다. 키 값은 출력하지 않고 존재 여부만 확인합니다.


In [1]:
# 고정한 패키지를 설치하고 비밀 키의 존재 여부만 확인합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3
from pathlib import Path
import os
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break

MODEL = "gpt-5.4-mini"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print({"langchain": version("langchain"), "langgraph": version("langgraph"), "model": MODEL, "cost_limit_usd": 1.0})


Note: you may need to restart the kernel to use updated packages.
{'langchain': '1.4.0', 'langgraph': '1.2.11', 'model': 'gpt-5.4-mini', 'cost_limit_usd': 1.0}


## 전문 에이전트를 도구로 감싸기

### 코드 읽기와 비교 경계

`@tool`은 Python 함수를 모델이 선택할 수 있는 도구 객체로 바꿉니다. 함수 이름과 docstring은 모델이 도구를 고를 때 읽는 설명이 되고, `question: str`은 입력 스키마가 됩니다. 중앙 `supervisor.invoke(...)`가 실행되면 모델이 먼저 `policy_specialist` 호출을 요청하고, LangChain이 함수를 실행한 뒤 반환 문자열을 `ToolMessage`로 대화에 넣습니다. 전문 에이전트의 `invoke`는 그 함수 안에서 별도의 모델 실행을 시작하므로 한 번의 위임에는 중앙 판단, 전문 처리, 중앙 종합이라는 호출 경계가 생길 수 있습니다.

서브에이전트와 일반 도구의 경계는 반환값을 누가 만드는지에 있습니다. 세율 조회처럼 입력과 출력이 결정적인 작업은 함수 도구가 더 분명합니다. 여러 자료를 읽고 판단 근거를 요약해야 하는 작업은 전문 프롬프트와 별도 문맥을 가진 서브에이전트가 도움이 됩니다. 이 예제의 문자 수 비교는 전달 범위를 줄였다는 사실만 보여 줍니다. 전문 에이전트가 더 정확해졌는지, 전체 토큰이 줄었는지, 고객 답변이 좋아졌는지는 별도의 질문 묶음과 평가 기준으로 측정해야 합니다.


In [2]:
# 교재용 고정 규정을 가진 전문 에이전트를 도구로 감쌉니다.
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

TRAINING_POLICY = """[교재용 환불 규정]
예정 배송일보다 3일 이상 지연되고 미배송 상태면 고객이 환불 검토를 요청할 수 있다.
담당자는 주문 번호와 현재 배송 상태를 확인한 뒤 환불 가능 여부를 안내한다."""
model = init_chat_model(f"openai:{MODEL}", reasoning_effort="none")
calls: list[str] = []
specialist = create_agent(model, tools=[], system_prompt="제공된 교재용 규정만 근거로 두 문장 이내로 답하세요.")

@tool
def policy_specialist(question: str) -> str:
    """교재용 환불 규정에서 근거를 확인합니다."""
    calls.append("policy_specialist")
    content = f"{TRAINING_POLICY}\n\n[고객 질문]\n{question}"
    result = specialist.invoke({"messages": [{"role": "user", "content": content}]})
    return result["messages"][-1].text

supervisor = create_agent(model, tools=[policy_specialist], system_prompt="환불 요청은 policy_specialist를 한 번 호출하고, 도구가 제공한 규정 범위 안에서만 답하세요.")


## 중앙 에이전트가 결과 종합

이 셀에서 중앙 에이전트는 교재용 고정 규정을 검토하는 전문 도구를 호출한 뒤 고객용 문장을 작성합니다. 출력의 환불 조건은 실습 규정 문자열 밖으로 확장되지 않아야 합니다.


In [3]:
# supervisor를 실제 실행하고 메시지 기록에서 도구 호출을 확인합니다.
request = "배송 지연 주문의 환불 가능 여부와 고객 안내 문구를 작성해 주세요."
result = supervisor.invoke({"messages": [{"role": "user", "content": request}]})
tool_names = [call["name"] for message in result["messages"] for call in getattr(message, "tool_calls", [])]
print("모델이 선택한 도구:", tool_names)
print("도구 실행 기록:", calls)
answer = result["messages"][-1].text
print("최종 답변:", answer)
assert tool_names == ["policy_specialist"] and calls == ["policy_specialist"]
assert "3일 이상" in answer and "주문 번호" in answer


모델이 선택한 도구: ['policy_specialist']
도구 실행 기록: ['policy_specialist']
최종 답변: 배송 지연 주문의 환불 가능 여부는 다음 기준으로 안내하시면 됩니다.

### 환불 가능 여부
- **예정 배송일보다 3일 이상 지연**되고,
- **아직 미배송 상태**인 경우에만  
  고객이 **환불 검토를 요청**할 수 있습니다.
- 담당자는 **주문 번호와 현재 배송 상태를 확인한 뒤** 환불 가능 여부를 안내해야 합니다.

### 고객 안내 문구 예시
“해당 주문은 **예정 배송일보다 3일 이상 지연**되고 **미배송 상태**인 경우에 한해 환불 검토가 가능합니다. **주문 번호와 현재 배송 상태를 확인한 후** 환불 가능 여부를 안내드리겠습니다.”

원하시면 제가 이 문구를 **더 공손한 고객센터 톤**이나 **짧은 문자/채팅용 문구**로도 바꿔드릴게요.


## 전달 맥락 줄이기

출력의 `전달_문자수` 38은 중앙 기록에서 분리한 고객 질문의 길이입니다. 전문 에이전트의 실제 입력은 이 질문에 `TRAINING_POLICY`를 별도로 결합합니다. 전문 답변의 정확성은 고정 규정과 답변을 항목별로 대조해 따로 검증해야 합니다.


In [4]:
# 전체 기록과 전문 에이전트에 전달한 질문의 문자 수를 비교합니다.
full_context = request + "\n" + "내부 기록 " * 100
focused_context = request
print({"전체_문자수": len(full_context), "전달_문자수": len(focused_context)})
assert len(focused_context) < len(full_context)


{'전체_문자수': 639, '전달_문자수': 38}
